In [ ]:
# Add these functions after your main code

def get_feature_importance_from_model(model, X, y, feature_names=None):
    
    if feature_names is None:
        feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
    
    # Train model if not already trained
    if not hasattr(model, 'feature_importances_'):
        model.fit(X, y)
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    return importance_df

def analyze_feature_importance_afterwards(model_params, X, y, feature_names=None, method='mean'):

    if feature_names is None:
        feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
    
    # Recreate the CV process to get feature importance
    cv = CombinatorialPurgedCV()
    cv_importances = []
    
    print("Re-running CV for feature importance analysis...")
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=1000,
            **model_params
        )
        
        model.fit(X_train, y_train, verbose=False)
        cv_importances.append(model.feature_importances_)
    
    # Aggregate importance
    if method == 'mean':
        avg_importance = np.mean(cv_importances, axis=0)
    else:
        avg_importance = np.median(cv_importances, axis=0)
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': avg_importance
    }).sort_values('importance', ascending=False)
    
    return importance_df

def plot_feature_importance(importance_df, top_n=20):

    top_features = importance_df.head(top_n)
    
    plt.figure(figsize=(12, 10))
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top {top_n} Feature Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== Top {top_n} Most Important Features ===")
    for i, row in top_features.iterrows():
        print(f"{row['feature']}: {row['importance']:.4f}")

def shap_analysis_afterwards(model_params, X, y, feature_names=None):

    try:
        import shap
        
        if feature_names is None:
            feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
        
        # Train final model
        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=1000,
            **model_params
        )
        model.fit(X, y)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        
        # Summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X, feature_names=feature_names, show=False)
        plt.title('SHAP Feature Importance')
        plt.tight_layout()
        plt.show()
        
        # Feature importance from SHAP
        shap_importance = np.abs(shap_values).mean(0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'shap_importance': shap_importance
        }).sort_values('shap_importance', ascending=False)
        
        print("\n=== SHAP Feature Importance (Top 15) ===")
        for i, row in importance_df.head(15).iterrows():
            print(f"{row['feature']}: {row['shap_importance']:.4f}")
            
        return importance_df
        
    except ImportError:
        print("SHAP not installed. Install with: pip install shap")
        return None

